反向互补与翻译

In [2]:
from Bio.Seq import Seq
import re
from Bio.SeqUtils import gc_fraction

1.DNA序列六框翻译

In [ ]:
"""
    对DNA序列进行六框翻译
    参数:
        dna_sequence: DNA序列字符串
    返回:
        字典: 6个阅读框的翻译结果
"""
def six_frame_translation(dna_sequence):
    seq = Seq(dna_sequence)
    results = {}
    # 正向链的三个阅读框（从第1、2、3个碱基开始）
    for frame in range(3):
        # 从frame位置开始翻译，每3个碱基一个密码子
        translated = seq[frame:].translate(to_stop=True)
        results[f"正向链 +{frame+1}"] = str(translated)

    # 反向互补链的三个阅读框
    rev_seq = seq.reverse_complement()
    for frame in range(3):
        translated = rev_seq[frame:].translate(to_stop=True)
        results[f"反向链 +{frame+1}"] = str(translated)

    return results
# 测试
test_dna = "ATGGCCCTGTGGATGCGCCTCCTGCCCGTGCTGC"
translations = six_frame_translation(test_dna)

for frame, protein in translations.items():
    print(f"{frame:12}: {protein}")

2.寻找最长开放阅读框（ORF）

In [4]:
def find_orfs(dna_sequence, min_length=100):
    """
    在DNA序列中查找开放阅读框
    参数:dna_sequence: DNA序列;min_length: 最小ORF长度（氨基酸数）
    返回:列表: 找到的ORF信息
    """
    seq = Seq(dna_sequence)
    orfs = []
    # 正向链
    for frame in range(3):
        trans = seq[frame:].translate(to_stop=True)
        # 在翻译结果中查找以M（甲硫氨酸，起始密码子）开头的片段
        for match in re.finditer(r'M[^*]*', str(trans)):
            if len(match.group()) >= min_length:
                orfs.append({
                    "strand": "+",
                    "frame": frame + 1,
                    "protein": match.group(),
                    "length_aa": len(match.group())
                })
    # 反向互补链
    rev_seq = seq.reverse_complement()
    for frame in range(3):
        trans = rev_seq[frame:].translate(to_stop=True)
        for match in re.finditer(r'M[^*]*', str(trans)):
            if len(match.group()) >= min_length:
                orfs.append({
                    "strand": "-",
                    "frame": frame + 1,
                    "protein": match.group(),
                    "length_aa": len(match.group())
                })

    return orfs

# 测试
test_dna = "ATGGCCCTGTGGATGCGCCTCCTGCCCGTGCTGTAAGCCGCCGCC"
orfs = find_orfs(test_dna, min_length=5)
for orf in orfs:
    print(f"链: {orf['strand']}, 阅读框: {orf['frame']}, 长度: {orf['length_aa']}aa")
    print(f"蛋白质: {orf['protein'][:50]}...\n")

链: +, 阅读框: 1, 长度: 11aa
蛋白质: MALWMRLLPVL...



3.整合之前编写的DNA分析器

In [ ]:
class DNAAnalyzer:
    """DNA序列分析器（增强版）"""
    def __init__(self, sequence):
        self.seq = Seq(sequence.upper())
        self.length = len(self.seq)
    def get_stats(self):
        """获取基本统计信息"""
        return {
            "length": self.length,
            "GC_content": round(gc_fraction(self.seq) * 100,2),
            "A_count": self.seq.count('A'),
            "T_count": self.seq.count('T'),
            "C_count": self.seq.count('C'),
            "G_count": self.seq.count('G')
        }
    def get_reverse_complement(self):
        """获取反向互补序列"""
        return str(self.seq.reverse_complement())
    def translate(self, table=1):
        """翻译为蛋白质（table=1是标准密码子表）"""
        return str(self.seq.translate(table=table, to_stop=True))
    def six_frame_translation(self):
        """六框翻译"""
        results = {}
        for frame in range(3):
            results[f"+{frame+1}"] = str(self.seq[frame:].translate(to_stop=True))
        rev_seq = self.seq.reverse_complement()
        for frame in range(3):
            results[f"-{frame+1}"] = str(rev_seq[frame:].translate(to_stop=True))
        return results

# 测试
analyzer = DNAAnalyzer("ATGGCCCTGTGGATGCGCCTCCTGCCCGTGCTG")
print(analyzer.get_stats())
print(f"反向互补: {analyzer.get_reverse_complement()}")
print(f"翻译: {analyzer.translate()}")